# 01 - Provision the Admin Monitoring archive Lakehouse

Creates or reuses a schema-enabled Lakehouse in the current workspace and initializes the extraction manifest, run audit, schema history, and reconciliation Delta tables.

This notebook is idempotent and can be rerun safely. It does not extract semantic-model data.


In [ ]:
target_workspace_name = "YOUR_TARGET_WORKSPACE"
lakehouse_name = "AdminMonitoringArchive"
data_schema = "admin_monitoring"
control_schema = "admin_monitoring_control"


In [ ]:
import time
from datetime import datetime, timezone

import requests
from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


def fabric_request(method, path, json_body=None, expected_statuses=(200, 201, 202)):
    token = notebookutils.credentials.getToken("pbi")
    response = requests.request(
        method,
        f"https://api.fabric.microsoft.com/v1/{path.lstrip('/')}",
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
            "x-ms-fabric-skill": "spark-cli",
        },
        json=json_body,
        timeout=120,
    )
    if response.status_code not in expected_statuses:
        raise RuntimeError(
            f"Fabric API {method} {path} returned HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )
    return response


def wait_for_operation(response, timeout_seconds=600):
    if response.status_code != 202:
        return
    operation_url = response.headers.get("Location")
    if not operation_url:
        raise RuntimeError("Fabric returned HTTP 202 without a Location header.")

    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        time.sleep(5)
        token = notebookutils.credentials.getToken("pbi")
        poll = requests.get(
            operation_url,
            headers={
                "Authorization": f"Bearer {token}",
                "x-ms-fabric-skill": "spark-cli",
            },
            timeout=120,
        )
        if not poll.ok:
            raise RuntimeError(f"Fabric operation poll failed: {poll.status_code} {poll.text[:2000]}")
        status = poll.json().get("status")
        if status == "Succeeded":
            return
        if status == "Failed":
            raise RuntimeError(f"Fabric operation failed: {poll.text[:2000]}")
    raise TimeoutError("Fabric operation did not complete within the configured timeout.")


def empty_delta(path, schema):
    spark.createDataFrame([], schema).write.format("delta").mode("ignore").save(path)


In [ ]:
context = notebookutils.runtime.context
workspace_id = str(context.get("currentWorkspaceId") or "")
workspace_name = str(context.get("currentWorkspaceName") or "")
if not workspace_id or workspace_name.casefold() != target_workspace_name.casefold():
    raise RuntimeError(
        f'This notebook must run in "{target_workspace_name}". Current workspace: "{workspace_name}".'
    )

lakehouses = fabric_request("GET", f"workspaces/{workspace_id}/lakehouses", expected_statuses=(200,)).json().get("value", [])
matches = [item for item in lakehouses if item.get("displayName", "").casefold() == lakehouse_name.casefold()]
if len(matches) > 1:
    raise RuntimeError(f'Multiple Lakehouses named "{lakehouse_name}" were found.')
if not matches:
    create_response = fabric_request(
        "POST",
        f"workspaces/{workspace_id}/items",
        {
            "displayName": lakehouse_name,
            "type": "Lakehouse",
            "description": "Durable archive of Microsoft-managed Admin monitoring semantic models.",
            "creationPayload": {"enableSchemas": True},
        },
    )
    wait_for_operation(create_response)
    lakehouses = fabric_request("GET", f"workspaces/{workspace_id}/lakehouses", expected_statuses=(200,)).json().get("value", [])
    matches = [item for item in lakehouses if item.get("displayName", "").casefold() == lakehouse_name.casefold()]
if len(matches) != 1:
    raise RuntimeError(f'Expected one Lakehouse named "{lakehouse_name}" after provisioning; found {len(matches)}.')

lakehouse_id = str(matches[0]["id"])
endpoint = notebookutils.conf.get("trident.onelake.endpoint").removeprefix("https://").removeprefix("http://").rstrip("/")
table_root = f"abfss://{workspace_id}@{endpoint}/{lakehouse_id}/Tables"
control_root = f"{table_root}/{control_schema}"
data_root = f"{table_root}/{data_schema}"

manifest = [
    ("Content Sharing", "Refresh Stats"),
    ("Content Sharing", "Changelog"),
    ("Content Sharing", "Workspace"),
    ("Content Sharing", "User"),
    ("Content Sharing", "Fabric Items"),
    ("Content Sharing", "Capacity"),
    ("Content Sharing", "Domain"),
    ("Feature Usage and Adoption", "Refresh Stats"),
    ("Feature Usage and Adoption", "Date"),
    ("Feature Usage and Adoption", "Changelog"),
    ("Feature Usage and Adoption", "Workspace"),
    ("Feature Usage and Adoption", "Audit"),
    ("Feature Usage and Adoption", "Capacity"),
    ("Feature Usage and Adoption", "User"),
    ("Feature Usage and Adoption", "Fabric Items"),
    ("OneLake Catalog - Governance for Admins", "Workspace"),
    ("OneLake Catalog - Governance for Admins", "User"),
    ("OneLake Catalog - Governance for Admins", "Fabric Items"),
    ("OneLake Catalog - Governance for Admins", "Capacity"),
    ("OneLake Catalog - Governance for Admins", "Domain"),
    ("OneLake Catalog - Governance for Admins", "Refresh Stats"),
    ("OneLake Catalog - Governance for Admins", "Audit"),
    ("OneLake Catalog - Governance for Admins", "Sensitivity Labels"),
    ("OneLake Catalog - Governance for Admins", "Subdomain"),
    ("OneLake Catalog - Governance for Admins", "Workspace Tags"),
    ("OneLake Catalog - Governance for Admins", "Item Tags"),
    ("OneLake Catalog - Governance for Admins", "Tags"),
    ("Purview Hub", "Fabric Items"),
    ("Purview Hub", "Workspace"),
    ("Purview Hub", "User"),
    ("Purview Hub", "Capacity"),
    ("Purview Hub", "Domain"),
]
manifest_df = spark.createDataFrame(manifest, ["semantic_model", "source_table"])
manifest_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{control_root}/table_manifest")
spark.createDataFrame(
    [(30, "30 days"), (90, "90 days"), (180, "180 days"), (365, "365 days")],
    ["days", "window"],
).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    f"{data_root}/dormancy_window"
)

run_audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("run_mode", StringType(), False),
    StructField("semantic_model", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("source_row_count", LongType(), True),
    StructField("extracted_row_count", LongType(), True),
    StructField("schema_hash", StringType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("started_utc", TimestampType(), False),
    StructField("ended_utc", TimestampType(), False),
])
schema_history_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("semantic_model", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("schema_hash", StringType(), False),
    StructField("columns_json", StringType(), False),
    StructField("recorded_utc", TimestampType(), False),
])
reconciliation_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("details", StringType(), True),
    StructField("checked_utc", TimestampType(), False),
])
empty_delta(f"{control_root}/run_audit", run_audit_schema)
empty_delta(f"{control_root}/schema_history", schema_history_schema)
empty_delta(f"{control_root}/reconciliation", reconciliation_schema)

display(spark.createDataFrame([
    (workspace_name, workspace_id, lakehouse_name, lakehouse_id, data_schema, control_schema, len(manifest))
], ["workspace", "workspace_id", "lakehouse", "lakehouse_id", "data_schema", "control_schema", "manifest_tables"]))
print("STEP 01 PASSED")
